In [48]:
import pandas as pd 
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import os
import re
import importlib

In [49]:
data_path = os.path.abspath("data/tcga_data2.csv")

raw_df = pd.read_csv(data_path, sep = '\t', low_memory = False)
raw_df.head(1)

,Gene,Study of Origin,Sample ID,Cancer Type,Cancer Type Detailed,Protein Change,Annotation,Custom Driver,Custom Driver Tiers,Functional Impact,...,Tumor Type,Used in Genomic Analysis,Vascular invasion indicator,Vessel Invasion,Vial number,Patient's Vital Status,Patient Weight,WGD,Winter Hypoxia Score,Year of Diagnosis
0,APC,"Colorectal Adenocarcinoma (TCGA, Firehose Legacy)",TCGA-AA-A010-01,Colorectal Cancer,Colon Adenocarcinoma,A2D,"OncoKB: Unknown, level NA, resistance NA;reVUE...",NaN,NaN,MutationAssessor: NA;SIFT: impact: deleterious...,...,NaN,NaN,NO,NaN,A,NaN,NaN,NaN,NaN,NaN


In [50]:
# https://docs.python.org/3/library/re.html
# \b is a space
cols = np.array(raw_df.columns.tolist())
pattern = re.compile(r'\bage\b', flags = re.IGNORECASE)
age_cols = np.array([])

for col in cols:
    # Regex needs pattern.search, instead of say "str".in(array)
    if pattern.search(col) and ("Diagnosis" in col):
        age_cols = np.append(age_cols, col)

age_cols

array(['Diagnosis Age', 'Age at Diagnosis', 'Age At Diagnosis'],
      dtype='<U32')

In [51]:
note_cols = ["Gene", "Sample ID", "Cancer Type Detailed", "Mutation Type", "Variant Type", "HGVSc", "MS", 
        "Protein Change", "Functional Impact"]

good_cols = np.concatenate((note_cols, age_cols))
df = raw_df[good_cols].copy()

# Gene expression as in email
# df = short_df[short_df["HGVSc"] == "ENST00000257430.4:c.835-8A>G"]
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6501 entries, 0 to 6500
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Gene                  6501 non-null   object 
 1   Sample ID             6501 non-null   object 
 2   Cancer Type Detailed  6501 non-null   object 
 3   Mutation Type         6501 non-null   object 
 4   Variant Type          6498 non-null   object 
 5   HGVSc                 5660 non-null   object 
 6   MS                    3188 non-null   object 
 7   Protein Change        6501 non-null   object 
 8   Functional Impact     6501 non-null   object 
 9   Diagnosis Age         3096 non-null   float64
 10  Age at Diagnosis      1477 non-null   float64
 11  Age At Diagnosis      304 non-null    float64
dtypes: float64(3), object(9)
memory usage: 609.6+ KB


In [52]:
# Stolen bfill from you! Axis = 1 looks across columns
df.loc[:, 'Age'] = df[age_cols].bfill(axis = 1).iloc[:, 0]
df = df.drop(columns = age_cols)

In [53]:
early_age = 50
df["Early Onset"] = df["Age"] < early_age
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6501 entries, 0 to 6500
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Gene                  6501 non-null   object 
 1   Sample ID             6501 non-null   object 
 2   Cancer Type Detailed  6501 non-null   object 
 3   Mutation Type         6501 non-null   object 
 4   Variant Type          6498 non-null   object 
 5   HGVSc                 5660 non-null   object 
 6   MS                    3188 non-null   object 
 7   Protein Change        6501 non-null   object 
 8   Functional Impact     6501 non-null   object 
 9   Age                   4877 non-null   float64
 10  Early Onset           6501 non-null   bool   
dtypes: bool(1), float64(1), object(9)
memory usage: 514.4+ KB


In [54]:
df = df.dropna(subset = ["Age"], axis = 0)
df = df.drop_duplicates(subset = ["Sample ID", "HGVSc"], keep = "first")

In [55]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4464 entries, 0 to 6500
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Gene                  4464 non-null   object 
 1   Sample ID             4464 non-null   object 
 2   Cancer Type Detailed  4464 non-null   object 
 3   Mutation Type         4464 non-null   object 
 4   Variant Type          4461 non-null   object 
 5   HGVSc                 4025 non-null   object 
 6   MS                    1308 non-null   object 
 7   Protein Change        4464 non-null   object 
 8   Functional Impact     4464 non-null   object 
 9   Age                   4464 non-null   float64
 10  Early Onset           4464 non-null   bool   
dtypes: bool(1), float64(1), object(9)
memory usage: 388.0+ KB


In [56]:
df.shape

(4464, 11)

In [57]:
df["Early Onset"].value_counts()

Early Onset
False    3586
True      878
Name: count, dtype: int64

In [58]:
df["Sample ID"]

0               TCGA-AA-A010-01
3               TCGA-EI-6917-01
4       coadread_dfci_2016_3235
5       coadread_dfci_2016_2367
6             P-0006759-T01-IM5
                 ...           
6494          P-0009842-T01-IM5
6495          P-0010238-T01-IM5
6496          P-0013828-T01-IM5
6499          P-0010742-T01-IM5
6500          P-0014026-T01-IM5
Name: Sample ID, Length: 4464, dtype: object

In [59]:
# Lists with the IDs of both, early onset and not early onset
IDs_Early_list = list(df["Sample ID"][df["Early Onset"]])
print(len(IDs_Early_list))
IDs_Not_Early_list = list(df["Sample ID"][df["Early Onset"] == False])
print(len(IDs_Not_Early_list))
print(IDs_Early_list)
print(IDs_Not_Early_list)

878
3586
['TCGA-AA-A010-01', 'TCGA-EI-6917-01', 'P-0010049-T01-IM5', 'P-0006365-T01-IM5', 'CRC812_NT', 'TCGA-AG-A002-01', 'P-0009971-T01-IM5', 'CRC642_NT', 'P-0012402-T01-IM5', 'P-0006202-T01-IM5', 'TCGA-AY-A8YK-01', 'P-0001702-T01-IM3', 'P-0006766-T01-IM5', 'CRC1032_NT', 'P-0047886-T01-IM6', 'P-0013788-T01-IM5', 'SER-SILU-CC-P0384-PT-01', 'CRC916_NT', '05CO011', 'P-0001424-T01-IM3', 'P-0001424-T02-IM3', 'P-0000687-T01-IM3', 'P-0001685-T02-IM3', 'P-0003437-T01-IM5', 'P-0006758-T01-IM5', 'P-0007180-T01-IM5', 'P-0007279-T01-IM5', 'P-0008063-T01-IM5', 'P-0013894-T01-IM5', 'CRC585_NT', 'CRC802_NT', 'CRC943_NT', 'CRC943_NT', 'TCGA-CK-4947-01', 'P-0001690-T01-IM3', 'P-0002770-T01-IM3', 'P-0003796-T01-IM5', 'P-0004159-T01-IM5', 'P-0005908-T01-IM5', 'P-0006075-T01-IM5', 'P-0006230-T01-IM5', 'P-0007092-T01-IM5', 'P-0007740-T01-IM5', 'P-0008317-T01-IM5', 'P-0008652-T01-IM5', 'P-0009690-T01-IM5', 'P-0010138-T01-IM5', 'P-0010207-T01-IM5', 'P-0014296-T01-IM6', 'CRC141_NT', 'CRC560_NT', 'CRC609_NT',

# Early Onset

In [60]:
import requests
import pandas as pd
import pandas as pd
from pyfaidx import Fasta # might need to do pip install first


base_url = "https://www.cbioportal.org/api"

# Choose study
profile = "crc_msk_2017_mutations"

# Choose sample ids the 35 IDs
sample_ids = IDs_Early_list

payload = {"sampleIds": sample_ids}

headers = {
    "Accept": "application/json",
    "Content-Type": "application/json"
}

r = requests.post(
    f"{base_url}/molecular-profiles/{profile}/mutations/fetch",
    json=payload,
    headers=headers
)

mut = pd.DataFrame(r.json())

mut.to_csv("all_mutations_crc_msk2017_samples_Early_Onset.csv", index=False)


# repeat if there are >1 studies, and merge into a single mut file

In [61]:
mut.head()

,uniqueSampleKey,uniquePatientKey,molecularProfileId,sampleId,patientId,entrezGeneId,studyId,center,mutationStatus,validationStatus,...,proteinChange,mutationType,ncbiBuild,variantType,keyword,chr,variantAllele,refseqMrnaId,proteinPosStart,proteinPosEnd
0,UC0wMDAwMjQxLVQwMS1JTTM6Y3JjX21za18yMDE3,UC0wMDAwMjQxOmNyY19tc2tfMjAxNw,crc_msk_2017_mutations,P-0000241-T01-IM3,P-0000241,324,crc_msk_2017,MSKCC,NA,NA,...,D1134Efs*3,Frame_Shift_Del,GRCh37,DEL,APC truncating,5,-,NM_000038.5,1134,1134
1,UC0wMDAwMjQxLVQwMS1JTTM6Y3JjX21za18yMDE3,UC0wMDAwMjQxOmNyY19tc2tfMjAxNw,crc_msk_2017_mutations,P-0000241-T01-IM3,P-0000241,324,crc_msk_2017,MSKCC,NA,NA,...,R2530W,Missense_Mutation,GRCh37,SNP,APC R2530 missense,5,T,NA,2530,2530
2,UC0wMDAwMjQxLVQwMS1JTTM6Y3JjX21za18yMDE3,UC0wMDAwMjQxOmNyY19tc2tfMjAxNw,crc_msk_2017_mutations,P-0000241-T01-IM3,P-0000241,324,crc_msk_2017,MSKCC,NA,NA,...,S1465Wfs*3,Frame_Shift_Del,GRCh37,DEL,APC truncating,5,-,NM_000038.5,1462,1465
3,UC0wMDAwMjQxLVQwMS1JTTM6Y3JjX21za18yMDE3,UC0wMDAwMjQxOmNyY19tc2tfMjAxNw,crc_msk_2017_mutations,P-0000241-T01-IM3,P-0000241,2065,crc_msk_2017,MSKCC,NA,NA,...,A232V,Missense_Mutation,GRCh37,SNP,ERBB3 A232 missense,12,T,NM_001982.3,232,232
4,UC0wMDAwMjQxLVQwMS1JTTM6Y3JjX21za18yMDE3,UC0wMDAwMjQxOmNyY19tc2tfMjAxNw,crc_msk_2017_mutations,P-0000241-T01-IM3,P-0000241,3845,crc_msk_2017,MSKCC,NA,NA,...,G12C,Missense_Mutation,GRCh37,SNP,KRAS G12 missense,12,A,NM_033360.2,12,12


In [62]:
mut.shape

(4362, 27)

In [63]:
mut[["sampleId", "mutationStatus", "referenceAllele", "proteinChange", "mutationType",
      "variantType", "keyword", "chr", "variantAllele"]].head()

,sampleId,mutationStatus,referenceAllele,proteinChange,mutationType,variantType,keyword,chr,variantAllele
0,P-0000241-T01-IM3,NA,CTATG,D1134Efs*3,Frame_Shift_Del,DEL,APC truncating,5,-
1,P-0000241-T01-IM3,NA,C,R2530W,Missense_Mutation,SNP,APC R2530 missense,5,T
2,P-0000241-T01-IM3,NA,AG,S1465Wfs*3,Frame_Shift_Del,DEL,APC truncating,5,-
3,P-0000241-T01-IM3,NA,C,A232V,Missense_Mutation,SNP,ERBB3 A232 missense,12,T
4,P-0000241-T01-IM3,NA,C,G12C,Missense_Mutation,SNP,KRAS G12 missense,12,A


In [64]:
import numpy as np
print(mut.columns)
print("Total recorded mutations:", len(mut))
grouped = mut[["sampleId", "mutationType", "referenceAllele"]].groupby("sampleId")#.count()
print("Total recorded patients:", len(grouped))
print("Average mutations recorded per sample:", np.mean(grouped.count()))
grouped.count()
# mut.head()

Index(['uniqueSampleKey', 'uniquePatientKey', 'molecularProfileId', 'sampleId',
       'patientId', 'entrezGeneId', 'studyId', 'center', 'mutationStatus',
       'validationStatus', 'tumorAltCount', 'tumorRefCount', 'normalAltCount',
       'normalRefCount', 'startPosition', 'endPosition', 'referenceAllele',
       'proteinChange', 'mutationType', 'ncbiBuild', 'variantType', 'keyword',
       'chr', 'variantAllele', 'refseqMrnaId', 'proteinPosStart',
       'proteinPosEnd'],
      dtype='object')
Total recorded mutations: 4362
Total recorded patients: 314
Average mutations recorded per sample: 13.89171974522293


,mutationType,referenceAllele
sampleId,,
P-0000241-T01-IM3,8,8
P-0000493-T01-IM3,4,4
P-0000511-T01-IM3,4,4
P-0000526-T01-IM3,6,6
P-0000561-T01-IM3,23,23
...,...,...
P-0014119-T01-IM5,6,6
P-0014168-T01-IM5,7,7
P-0014195-T01-IM6,6,6


In [65]:
import pandas as pd
from pyfaidx import Fasta

# -----------------------------
# Inputs
# -----------------------------
mut = pd.read_csv("all_mutations_crc_msk2017_samples_Early_Onset.csv")

# Change this path if needed
fasta_path = "data/hg19.fa"
ref = Fasta(fasta_path)

# -----------------------------
# Basic cleanup / added columns
# -----------------------------
mut = mut.copy()

# cBioPortal export uses 'keyword' as gene symbol here
mut["gene"] = mut["keyword"]

# HGVSc is not present in this export
mut["HGVSc"] = pd.NA

# Mark the specific target splice-creating mutation:
# chr5:112151184 A>G
mut["target splice"] = (
    mut["chr"].astype(str).str.replace(r"\.0$", "", regex=True).isin(["5", "chr5"]) &
    pd.to_numeric(mut["startPosition"], errors="coerce").eq(112151184) &
    mut["referenceAllele"].astype(str).str.upper().eq("A") &
    mut["variantAllele"].astype(str).str.upper().eq("G")
).astype(int)

# -----------------------------
# Keep all single-base substitutions
# This includes splice SNVs and splice-creating SNVs
# -----------------------------
is_sbs = (
    mut["referenceAllele"].astype(str).str.len().eq(1) &
    mut["variantAllele"].astype(str).str.len().eq(1) &
    mut["referenceAllele"].astype(str).str.upper().isin(list("ACGT")) &
    mut["variantAllele"].astype(str).str.upper().isin(list("ACGT"))
)

snv = mut.loc[is_sbs].copy()

# -----------------------------
# Helpers for COSMIC96 assignment
# -----------------------------
comp = str.maketrans("ACGT", "TGCA")

def revcomp(seq):
    return seq.translate(comp)[::-1]

def clean_chrom(chrom):
    chrom = str(chrom).strip()
    if chrom.endswith(".0"):
        chrom = chrom[:-2]
    return chrom

def get_context_pyfaidx(ref, chrom, pos1):
    chrom = clean_chrom(chrom)

    candidates = [chrom]
    if chrom.startswith("chr"):
        candidates.append(chrom.replace("chr", "", 1))
    else:
        candidates.append("chr" + chrom)

    used_chrom = None
    for c in candidates:
        if c in ref.keys():
            used_chrom = c
            break

    if used_chrom is None:
        raise KeyError(f"Chromosome {chrom} not found in FASTA")

    pos1 = int(pos1)

    # pyfaidx uses 0-based slicing, end-exclusive
    context = ref[used_chrom][pos1 - 2 : pos1 + 1].seq.upper()

    if len(context) != 3:
        return None, used_chrom

    return context, used_chrom

def to_cosmic96(context, ref_base, alt_base):
    context = context.upper()
    ref_base = ref_base.upper()
    alt_base = alt_base.upper()

    if len(context) != 3:
        return None

    if context[1] != ref_base:
        return None

    # Normalize to pyrimidine representation
    if ref_base in ["A", "G"]:
        context = revcomp(context)
        ref_base = revcomp(ref_base)
        alt_base = revcomp(alt_base)

    if ref_base not in ["C", "T"]:
        return None

    return f"{context[0]}[{ref_base}>{alt_base}]{context[2]}"

# -----------------------------
# Annotate trinucleotide context and COSMIC96
# -----------------------------
contexts = []
classes = []
problems = []
used_chrs = []

for _, row in snv.iterrows():
    try:
        context, used_chr = get_context_pyfaidx(ref, row["chr"], row["startPosition"])
        ref_base = str(row["referenceAllele"]).upper()
        alt_base = str(row["variantAllele"]).upper()

        cosmic = to_cosmic96(context, ref_base, alt_base)

        contexts.append(context)
        classes.append(cosmic)
        used_chrs.append(used_chr)

        if context is None:
            problems.append("Could not extract 3bp context")
        elif cosmic is None:
            problems.append(
                f"Reference allele mismatch: FASTA middle base={context[1]}, mutation ref={ref_base}"
            )
        else:
            problems.append(None)

    except Exception as e:
        contexts.append(None)
        classes.append(None)
        used_chrs.append(None)
        problems.append(str(e))

snv["used_chr"] = used_chrs
snv["trinuc_context"] = contexts
snv["COSMIC96"] = classes
snv["problem"] = problems

# -----------------------------
# Build final dataframe
# -----------------------------
final_df = snv[[
    "sampleId",
    "patientId",
    "gene",
    "HGVSc",
    "target splice",
    "chr",
    "used_chr",
    "startPosition",
    "endPosition",
    "referenceAllele",
    "variantAllele",
    "proteinChange",
    "mutationType",
    "variantType",
    "ncbiBuild",
    "trinuc_context",
    "COSMIC96",
    "problem"
]].copy()

# -----------------------------
# Save outputs
# -----------------------------
final_df.to_csv("all_mutations_crc_msk2017_samples_Early_Onset_cosmic96.csv", index=False)

counts = final_df["COSMIC96"].value_counts(dropna=True).sort_index()
counts.to_csv("Early_Onset_cosmic96_counts.csv", header=["count"])

# -----------------------------
# Checks
# -----------------------------
print(final_df.head(20))

print("\nTarget splice rows:")
print(
    final_df.loc[final_df["target splice"] == 1, [
        "sampleId",
        "gene",
        "proteinChange",
        "mutationType",
        "chr",
        "startPosition",
        "referenceAllele",
        "variantAllele",
        "trinuc_context",
        "COSMIC96",
        "problem"
    ]]
)


             sampleId  patientId                  gene HGVSc  target splice  \
1   P-0000241-T01-IM3  P-0000241    APC R2530 missense  <NA>              0   
3   P-0000241-T01-IM3  P-0000241   ERBB3 A232 missense  <NA>              0   
4   P-0000241-T01-IM3  P-0000241     KRAS G12 missense  <NA>              0   
5   P-0000241-T01-IM3  P-0000241    PAK1 L470 missense  <NA>              0   
6   P-0000241-T01-IM3  P-0000241  PTCH1 V1418 missense  <NA>              0   
7   P-0000241-T01-IM3  P-0000241    TP53 R158 missense  <NA>              0   
8   P-0000493-T01-IM3  P-0000493        APC truncating  <NA>              0   
9   P-0000493-T01-IM3  P-0000493  PIK3CA C901 missense  <NA>              0   
10  P-0000493-T01-IM3  P-0000493      PTCH1 truncating  <NA>              0   
11  P-0000493-T01-IM3  P-0000493    TP53 R248 missense  <NA>              0   
13  P-0000511-T01-IM3  P-0000511    BRAF K601 missense  <NA>              0   
14  P-0000511-T01-IM3  P-0000511    TP53 R248 missen

In [66]:
print(final_df.columns)
print("Total recorded mutations:", len(final_df))
grouped = final_df[["sampleId", "mutationType", "referenceAllele"]].groupby("sampleId")#.count()
print("Total recorded patients:", len(grouped))
print("Average mutations recorded per sample:", np.mean(grouped.count()))
grouped.count()
# mut.head()

Index(['sampleId', 'patientId', 'gene', 'HGVSc', 'target splice', 'chr',
       'used_chr', 'startPosition', 'endPosition', 'referenceAllele',
       'variantAllele', 'proteinChange', 'mutationType', 'variantType',
       'ncbiBuild', 'trinuc_context', 'COSMIC96', 'problem'],
      dtype='object')
Total recorded mutations: 3509
Total recorded patients: 313
Average mutations recorded per sample: 11.210862619808307


,mutationType,referenceAllele
sampleId,,
P-0000241-T01-IM3,6,6
P-0000493-T01-IM3,4,4
P-0000511-T01-IM3,3,3
P-0000526-T01-IM3,6,6
P-0000561-T01-IM3,18,18
...,...,...
P-0014119-T01-IM5,5,5
P-0014168-T01-IM5,6,6
P-0014195-T01-IM6,6,6


# Not Early Onset

In [67]:
import requests
import pandas as pd
import pandas as pd
from pyfaidx import Fasta # might need to do pip install first


base_url = "https://www.cbioportal.org/api"

# Choose study
profile = "crc_msk_2017_mutations"

# Choose sample ids the 35 IDs
sample_ids = IDs_Not_Early_list

payload = {"sampleIds": sample_ids}

headers = {
    "Accept": "application/json",
    "Content-Type": "application/json"
}

r = requests.post(
    f"{base_url}/molecular-profiles/{profile}/mutations/fetch",
    json=payload,
    headers=headers
)

mut = pd.DataFrame(r.json())

mut.to_csv("all_mutations_crc_msk2017_samples_Not_Early_Onset.csv", index=False)


# repeat if there are >1 studies, and merge into a single mut file

In [68]:
mut.head()

,uniqueSampleKey,uniquePatientKey,molecularProfileId,sampleId,patientId,entrezGeneId,studyId,center,mutationStatus,validationStatus,...,proteinChange,mutationType,ncbiBuild,variantType,keyword,chr,variantAllele,refseqMrnaId,proteinPosStart,proteinPosEnd
0,UC0wMDAwMTE5LVQwMS1JTTM6Y3JjX21za18yMDE3,UC0wMDAwMTE5OmNyY19tc2tfMjAxNw,crc_msk_2017_mutations,P-0000119-T01-IM3,P-0000119,324,crc_msk_2017,MSKCC,NA,NA,...,E1097*,Nonsense_Mutation,GRCh37,SNP,APC truncating,5,T,NM_000038.5,1097,1097
1,UC0wMDAwMTE5LVQwMS1JTTM6Y3JjX21za18yMDE3,UC0wMDAwMTE5OmNyY19tc2tfMjAxNw,crc_msk_2017_mutations,P-0000119-T01-IM3,P-0000119,324,crc_msk_2017,MSKCC,NA,NA,...,T1438Yfs*35,Frame_Shift_Del,GRCh37,DEL,APC truncating,5,T,NM_000038.5,1438,1438
2,UC0wMDAwMTE5LVQwMS1JTTM6Y3JjX21za18yMDE3,UC0wMDAwMTE5OmNyY19tc2tfMjAxNw,crc_msk_2017_mutations,P-0000119-T01-IM3,P-0000119,2044,crc_msk_2017,MSKCC,NA,NA,...,R37W,Missense_Mutation,GRCh37,SNP,EPHA5 R37 missense,4,A,NM_004439.5,37,37
3,UC0wMDAwMTE5LVQwMS1JTTM6Y3JjX21za18yMDE3,UC0wMDAwMTE5OmNyY19tc2tfMjAxNw,crc_msk_2017_mutations,P-0000119-T01-IM3,P-0000119,3845,crc_msk_2017,MSKCC,NA,NA,...,A146T,Missense_Mutation,GRCh37,SNP,KRAS A146 missense,12,T,NM_033360.2,146,146
4,UC0wMDAwMTE5LVQwMS1JTTM6Y3JjX21za18yMDE3,UC0wMDAwMTE5OmNyY19tc2tfMjAxNw,crc_msk_2017_mutations,P-0000119-T01-IM3,P-0000119,4089,crc_msk_2017,MSKCC,NA,NA,...,P91L,Missense_Mutation,GRCh37,SNP,SMAD4 P91 missense,18,T,NM_005359.5,91,91


In [69]:
mut.shape

(6990, 27)

In [70]:
mut[["sampleId", "mutationStatus", "referenceAllele", "proteinChange", "mutationType",
      "variantType", "keyword", "chr", "variantAllele"]].head()

,sampleId,mutationStatus,referenceAllele,proteinChange,mutationType,variantType,keyword,chr,variantAllele
0,P-0000119-T01-IM3,NA,G,E1097*,Nonsense_Mutation,SNP,APC truncating,5,T
1,P-0000119-T01-IM3,NA,AC,T1438Yfs*35,Frame_Shift_Del,DEL,APC truncating,5,T
2,P-0000119-T01-IM3,NA,G,R37W,Missense_Mutation,SNP,EPHA5 R37 missense,4,A
3,P-0000119-T01-IM3,NA,C,A146T,Missense_Mutation,SNP,KRAS A146 missense,12,T
4,P-0000119-T01-IM3,NA,C,P91L,Missense_Mutation,SNP,SMAD4 P91 missense,18,T


In [71]:
import numpy as np
print(mut.columns)
print("Total recorded mutations:", len(mut))
grouped = mut[["sampleId", "mutationType", "referenceAllele"]].groupby("sampleId")#.count()
print("Total recorded patients:", len(grouped))
print("Average mutations recorded per sample:", np.mean(grouped.count()))
grouped.count()
# mut.head()

Index(['uniqueSampleKey', 'uniquePatientKey', 'molecularProfileId', 'sampleId',
       'patientId', 'entrezGeneId', 'studyId', 'center', 'mutationStatus',
       'validationStatus', 'tumorAltCount', 'tumorRefCount', 'normalAltCount',
       'normalRefCount', 'startPosition', 'endPosition', 'referenceAllele',
       'proteinChange', 'mutationType', 'ncbiBuild', 'variantType', 'keyword',
       'chr', 'variantAllele', 'refseqMrnaId', 'proteinPosStart',
       'proteinPosEnd'],
      dtype='object')
Total recorded mutations: 6990
Total recorded patients: 554
Average mutations recorded per sample: 12.617328519855596


,mutationType,referenceAllele
sampleId,,
P-0000119-T01-IM3,9,9
P-0000520-T01-IM3,5,5
P-0000552-T01-IM3,6,6
P-0000625-T01-IM3,9,9
P-0000635-T01-IM3,5,5
...,...,...
P-0014116-T01-IM5,12,12
P-0014133-T01-IM5,6,6
P-0014182-T01-IM6,9,9


In [72]:
import pandas as pd
from pyfaidx import Fasta

# -----------------------------
# Inputs
# -----------------------------
mut = pd.read_csv("all_mutations_crc_msk2017_samples_Not_Early_Onset.csv")

# Change this path if needed
fasta_path = "data/hg19.fa"
ref = Fasta(fasta_path)

# -----------------------------
# Basic cleanup / added columns
# -----------------------------
mut = mut.copy()

# cBioPortal export uses 'keyword' as gene symbol here
mut["gene"] = mut["keyword"]

# HGVSc is not present in this export
mut["HGVSc"] = pd.NA

# Mark the specific target splice-creating mutation:
# chr5:112151184 A>G
mut["target splice"] = (
    mut["chr"].astype(str).str.replace(r"\.0$", "", regex=True).isin(["5", "chr5"]) &
    pd.to_numeric(mut["startPosition"], errors="coerce").eq(112151184) &
    mut["referenceAllele"].astype(str).str.upper().eq("A") &
    mut["variantAllele"].astype(str).str.upper().eq("G")
).astype(int)

# -----------------------------
# Keep all single-base substitutions
# This includes splice SNVs and splice-creating SNVs
# -----------------------------
is_sbs = (
    mut["referenceAllele"].astype(str).str.len().eq(1) &
    mut["variantAllele"].astype(str).str.len().eq(1) &
    mut["referenceAllele"].astype(str).str.upper().isin(list("ACGT")) &
    mut["variantAllele"].astype(str).str.upper().isin(list("ACGT"))
)

snv = mut.loc[is_sbs].copy()

# -----------------------------
# Helpers for COSMIC96 assignment
# -----------------------------
comp = str.maketrans("ACGT", "TGCA")

# -----------------------------
# Annotate trinucleotide context and COSMIC96
# -----------------------------
contexts = []
classes = []
problems = []
used_chrs = []

for _, row in snv.iterrows():
    try:
        context, used_chr = get_context_pyfaidx(ref, row["chr"], row["startPosition"])
        ref_base = str(row["referenceAllele"]).upper()
        alt_base = str(row["variantAllele"]).upper()

        cosmic = to_cosmic96(context, ref_base, alt_base)

        contexts.append(context)
        classes.append(cosmic)
        used_chrs.append(used_chr)

        if context is None:
            problems.append("Could not extract 3bp context")
        elif cosmic is None:
            problems.append(
                f"Reference allele mismatch: FASTA middle base={context[1]}, mutation ref={ref_base}"
            )
        else:
            problems.append(None)

    except Exception as e:
        contexts.append(None)
        classes.append(None)
        used_chrs.append(None)
        problems.append(str(e))

snv["used_chr"] = used_chrs
snv["trinuc_context"] = contexts
snv["COSMIC96"] = classes
snv["problem"] = problems

# -----------------------------
# Build final dataframe
# -----------------------------
final_df = snv[[
    "sampleId",
    "patientId",
    "gene",
    "HGVSc",
    "target splice",
    "chr",
    "used_chr",
    "startPosition",
    "endPosition",
    "referenceAllele",
    "variantAllele",
    "proteinChange",
    "mutationType",
    "variantType",
    "ncbiBuild",
    "trinuc_context",
    "COSMIC96",
    "problem"
]].copy()

# -----------------------------
# Save outputs
# -----------------------------
final_df.to_csv("all_mutations_crc_msk2017_samples_Not_Early_Onset_cosmic96.csv", index=False)

counts = final_df["COSMIC96"].value_counts(dropna=True).sort_index()
counts.to_csv("Not_Early_Onset_cosmic96_counts.csv", header=["count"])

# -----------------------------
# Checks
# -----------------------------
print(final_df.head(20))

print("\nTarget splice rows:")
print(
    final_df.loc[final_df["target splice"] == 1, [
        "sampleId",
        "gene",
        "proteinChange",
        "mutationType",
        "chr",
        "startPosition",
        "referenceAllele",
        "variantAllele",
        "trinuc_context",
        "COSMIC96",
        "problem"
    ]]
)

             sampleId  patientId                   gene HGVSc  target splice  \
0   P-0000119-T01-IM3  P-0000119         APC truncating  <NA>              0   
2   P-0000119-T01-IM3  P-0000119     EPHA5 R37 missense  <NA>              0   
3   P-0000119-T01-IM3  P-0000119     KRAS A146 missense  <NA>              0   
4   P-0000119-T01-IM3  P-0000119     SMAD4 P91 missense  <NA>              0   
6   P-0000119-T01-IM3  P-0000119    HNF1A V617 missense  <NA>              0   
7   P-0000119-T01-IM3  P-0000119    STAG2 L503 missense  <NA>              0   
8   P-0000119-T01-IM3  P-0000119   KMT2C G2346 missense  <NA>              0   
10  P-0000520-T01-IM3  P-0000520         APC truncating  <NA>              0   
11  P-0000520-T01-IM3  P-0000520      KDR V514 missense  <NA>              0   
12  P-0000520-T01-IM3  P-0000520  PIK3CA H1047 missense  <NA>              0   
13  P-0000520-T01-IM3  P-0000520        SOX9 truncating  <NA>              0   
14  P-0000552-T01-IM3  P-0000552        

In [73]:
print(final_df.columns)
print("Total recorded mutations:", len(final_df))
grouped = final_df[["sampleId", "mutationType", "referenceAllele"]].groupby("sampleId")#.count()
print("Total recorded patients:", len(grouped))
print("Average mutations recorded per sample:", np.mean(grouped.count()))
grouped.count()
# mut.head()

Index(['sampleId', 'patientId', 'gene', 'HGVSc', 'target splice', 'chr',
       'used_chr', 'startPosition', 'endPosition', 'referenceAllele',
       'variantAllele', 'proteinChange', 'mutationType', 'variantType',
       'ncbiBuild', 'trinuc_context', 'COSMIC96', 'problem'],
      dtype='object')
Total recorded mutations: 5761
Total recorded patients: 552
Average mutations recorded per sample: 10.43659420289855


,mutationType,referenceAllele
sampleId,,
P-0000119-T01-IM3,7,7
P-0000520-T01-IM3,4,4
P-0000552-T01-IM3,6,6
P-0000625-T01-IM3,8,8
P-0000635-T01-IM3,5,5
...,...,...
P-0014116-T01-IM5,11,11
P-0014133-T01-IM5,5,5
P-0014182-T01-IM6,8,8
